In [1]:
"""
- ID3 (Iterative Dichotomiser 3) was developed in 1986 by Ross Quinlan. The algorithm creates a multiway tree, finding 
      for each node (i.e. in a greedy manner) the categorical feature that will yield the largest information gain for 
      categorical targets. Trees are grown to their maximum size and then a pruning step is usually applied to improve 
      the ability of the tree to generalise to unseen data.

- C4.5 is the successor to ID3 and removed the restriction that features must be categorical by dynamically defining a 
      discrete attribute (based on numerical variables) that partitions the continuous attribute value into a discrete 
      set of intervals. C4.5 converts the trained trees (i.e. the output of the ID3 algorithm) into sets of if-then rules.
      These accuracy of each rule is then evaluated to determine the order in which they should be applied. Pruning is 
      done by removing a rule’s precondition if the accuracy of the rule improves without it.

- C5.0 is Quinlan’s latest version release under a proprietary license. It uses less memory and builds smaller rulesets 
      than C4.5 while being more accurate.

- CART (Classification and Regression Trees) is very similar to C4.5, but it differs in that it supports numerical 
      target variables (regression) and does not compute rule sets. CART constructs binary trees using the feature and 
      threshold that yield the largest information gain at each node.

*** scikit-learn uses an optimised version of the CART algorithm ***
Documentation: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
"""

%matplotlib inline
import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("error", category=UndefinedMetricWarning)
warnings.filterwarnings(action="ignore",category=DeprecationWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer
from sklearn.metrics import precision_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import fbeta_score
from sklearn.metrics import auc
from sklearn.model_selection import StratifiedKFold
from sklearn import metrics
from sklearn.tree import export_graphviz
from sklearn.externals.six import StringIO  
from IPython.display import Image  
import pydotplus
import pickle

def PlotTree(clf, features, name="Tree.png"):
    dot_data=StringIO()
    export_graphviz(clf, out_file=dot_data, filled=True, rounded=True, special_characters=True,
                    feature_names=features,class_names=['0.0','1.0'])
    graph = pydotplus.graph_from_dot_data(dot_data.getvalue())  
    graph.write_png(name)
    return Image(graph.create_png())

def GetMetrics(y_test, y_pred):
    print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
    display("confusion_matrix", pd.DataFrame(metrics.confusion_matrix(y_test, y_pred)))
    report = metrics.classification_report(y_test, y_pred, output_dict=True)
    display("metrics", pd.DataFrame(report).transpose())


In [2]:
## Load Dataset ....

Class=["refatoracao"]
features = [ "LOC", "NPM", "WMC", "NOC", "DIT", "CAM", 
             "CBO", "RFC", "CA", "CE", "LCOM3", #"LCOM",
             "DAM", "MOA", "MFA", "IC", "CBM", "AMC", 
           ]
df = pd.read_csv('dataset_refat_final.csv',delimiter=";", decimal=".") #, header=True) #,names=features)
x = df[features]
y = df[Class]
print("Data Size: ",df.shape)
df[features+Class].head()

Data Size:  (2030, 25)


,LOC,NPM,WMC,NOC,DIT,CAM,CBO,RFC,CA,CE,LCOM3,DAM,MOA,MFA,IC,CBM,AMC,refatoracao
0,1656.0,88.0,100.0,0.0,0.0,0.2148,6.0,140.0,0.0,6.0,0.2222,0.8333,0.0,0.0,0.0,0.0,15.5,1.0
1,253.0,11.0,14.0,4.0,1.0,0.1688,7.0,46.0,4.0,3.0,0.8718,0.3333,0.0,0.0,0.0,0.0,168571.0,1.0
2,1421.0,88.0,96.0,0.0,0.0,0.2211,6.0,130.0,0.0,6.0,0.5263,0.5000,0.0,0.0,0.0,0.0,137813.0,1.0
3,517.0,24.0,30.0,0.0,0.0,0.2867,5.0,57.0,0.0,5.0,0.9151,0.0000,0.0,0.0,0.0,0.0,15.8,1.0
4,783.0,17.0,22.0,0.0,0.0,0.2273,8.0,65.0,0.0,8.0,0.9568,0.1875,1.0,0.0,0.0,0.0,331364.0,1.0


In [3]:
## Find the best params for DecisionTreeClassifier and we use KFold

dec_tree = DecisionTreeClassifier(random_state=0) # random_state - deterministic behaviour during fitting random_state has to be fixed to an integer
pipe = Pipeline(steps=[ ('dec_tree', dec_tree),
                      ])
parameters = dict(dec_tree__criterion=['gini', 'entropy'],
                  dec_tree__min_samples_split=[16, 32, 64, 128, 256],
                  dec_tree__min_samples_leaf=[16, 32, 64, 128, 256],
                  dec_tree__max_depth=[2, 4, 6, 8, 10, 12, 14, 16, 18],
                  dec_tree__max_features=list(range(1,len(features)+1)),
                 )
scores = {'accuracy' :make_scorer(accuracy_score),
          'recall'   :make_scorer(recall_score),
          'precision':make_scorer(precision_score),
          'f1'       :make_scorer(fbeta_score, beta = 1),
          'auc'      :make_scorer(auc),
         }
KF = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
clf_GS = GridSearchCV(pipe, param_grid=parameters, n_jobs=-1, cv=KF,
                      #scoring = scores,   refit = 'auc', 
                      scoring = 'roc_auc',
                     )

# split in Train_Test (70%) and Validation (30%) (train, test, validation, prevent overfitting)
x_train_test, x_validation, y_train_test, y_validation = train_test_split(x, y, test_size=0.3, random_state=0) 
# KFold will split Train_Test into 10-Folds: 9 to Train and 1 to Test. Next, GridSearchCV will find the best model
clf_GS_fit=clf_GS.fit(x_train_test, y_train_test) 

pickle.dump(clf_GS, open("BestDecisionTree", 'wb')) # Save Model

print("Decision tree tuning")
print("Steps:\n",pipe.steps)
print("Best Parameters:\n\t Params: {}\n\t Score: {}\n".format(clf_GS.best_params_,clf_GS.best_score_))

print("** metrics for Validation data set")
y_pred = clf_GS_fit.predict(x_validation)
GetMetrics(y_validation, y_pred)

img=PlotTree(clf_GS.best_estimator_.named_steps["dec_tree"], features, "Tree_GS.png")


Decision tree tuning
Steps:
 [('dec_tree', DecisionTreeClassifier(class_weight=None, criterion='gini', max_depth=None,
            max_features=None, max_leaf_nodes=None,
            min_impurity_decrease=0.0, min_impurity_split=None,
            min_samples_leaf=1, min_samples_split=2,
            min_weight_fraction_leaf=0.0, presort=False, random_state=0,
            splitter='best'))]
Best Parameters:
	 Params: {'dec_tree__criterion': 'entropy', 'dec_tree__max_depth': 6, 'dec_tree__max_features': 14, 'dec_tree__min_samples_leaf': 64, 'dec_tree__min_samples_split': 16}
	 Score: 0.9282872797815904

** metrics for Validation data set
Accuracy: 0.8538587848932676


'confusion_matrix'

,0,1
0,232,70
1,19,288


'metrics'

,f1-score,precision,recall,support
0.0,0.839060,0.924303,0.768212,302.0
1.0,0.866165,0.804469,0.938111,307.0
micro avg,0.853859,0.853859,0.853859,609.0
macro avg,0.852613,0.864386,0.853161,609.0
weighted avg,0.852724,0.863894,0.853859,609.0


In [4]:
TrainParmResults=pd.DataFrame(clf_GS.cv_results_)
pd.options.display.max_columns = None
#TrainParmResults.head(6) # -1 Show all 

In [5]:
%matplotlib inline
import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("error", category=UndefinedMetricWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_validate
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedKFold
from collections import OrderedDict
import pandas as pd

# https://sklearn-template.readthedocs.io/en/latest/user_guide.html
class PaperTreeModel(BaseEstimator, ClassifierMixin):
    def __init__(self):
        pass

    def fit(self, x, y):
        return self
    
    def CreateOut(self, data, out):
        if len(data)>0:
            index=data.index.values.tolist()
            tmp=dict.fromkeys(index , out)
            self.out_={**self.out_, **tmp}
        
    def predict(self, x): # Paper model
        self.out_={}
        self.CreateOut(x[ (x["WMC"] >9.500) ], 1.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]>597) & (x["WMC"]>8.50) ], 1.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]>597) & (x["WMC"]<=8.50) ], 0.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]<=597) & (x["NPM"]>7.50) & (x["CAM"]>0.270) & (x["NOC"]>0.50) & (x["NOC"]>1.50) ], 0.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]<=597) & (x["NPM"]>7.50) & (x["CAM"]>0.270) & (x["NOC"]>0.50) & (x["NOC"]<=1.50) ], 1.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]<=597) & (x["NPM"]>7.50) & (x["CAM"]>0.270) & (x["NOC"]<=0.50) & (x["LOC"]>48) ], 0.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]<=597) & (x["NPM"]>7.50) & (x["CAM"]>0.270) & (x["NOC"]<=0.50) & (x["LOC"]<=48) & (x["DIT"]>0.50) ], 0.0)        
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]<=597) & (x["NPM"]>7.50) & (x["CAM"]>0.270) & (x["NOC"]<=0.50) & (x["LOC"]<=48) & (x["DIT"]<=0.50) ], 1.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]<=597) & (x["NPM"]>7.50) & (x["CAM"]>0.270) ], 1.0)
        self.CreateOut(x[ (x["WMC"]<=9.500) & (x["LOC"]<=597) & (x["NPM"]<=7.50) ], 0.0)
        self.CreateOut(x[~x.index.isin(list(self.out_.keys()))], 1.0 )
        self.out_ = OrderedDict(sorted(self.out_.items()))
        return list(self.out_.values())


## Load Dataset ....
Class=["refatoracao"]
features = [ "LOC", "NPM", "WMC", "NOC", "DIT", "CAM", 
             "CBO", "RFC", "CA", "CE", "LCOM3", #"LCOM",
             "DAM", "MOA", "MFA", "IC", "CBM", "AMC", ]
df = pd.read_csv('dataset_refat_final.csv',delimiter=";", decimal=".") #, header=True) #,names=features)
x = df[features]
y = df[Class]
print("Data Size: ",df.shape)
df[features+Class].head()

# Test model ....
clf=PaperTreeModel()
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
scoring = ['precision', 'recall', 'f1']
scores = cross_validate(clf, x, y, cv=cv, scoring=scoring)
display(pd.DataFrame(scores))

Data Size:  (2030, 25)


,fit_time,score_time,test_precision,train_precision,test_recall,train_recall,test_f1,train_f1
0,0.0,0.037817,0.792308,0.766497,1.000000,0.985854,0.884120,0.862446
1,0.0,0.053409,0.778626,0.767993,0.990291,0.986942,0.871795,0.863810
2,0.0,0.046893,0.785714,0.767285,0.970588,0.989130,0.868421,0.864198
3,0.0,0.046877,0.765152,0.769492,0.990196,0.986957,0.863248,0.864762
4,0.0,0.046829,0.757576,0.770339,0.980392,0.988043,0.854701,0.865714
5,0.0,0.046859,0.744361,0.771841,0.970588,0.989130,0.842553,0.867080
6,0.0,0.046870,0.796875,0.766047,1.000000,0.985870,0.886957,0.862167
7,0.0,0.046890,0.744526,0.771915,1.000000,0.985870,0.853556,0.865871
8,0.0,0.046838,0.793651,0.766442,0.980392,0.988043,0.877193,0.863248
9,0.0,0.068997,0.737226,0.772766,0.990196,0.986957,0.845188,0.866826
